# Garak Artifact Demo

This notebook takes a **scenario** YAML, labels it for Garak coverage (`full` / `partial` / `skip`), then generates the **Garak probe artifact** (chat history + detector rubrics for LLM-as-a-judge).

**Pipeline:**
1. Select a scenario
2. Build schema `EnvironmentSpec` → `runs/{id}/spec.json`
3. Label via `gate_garak` → `full` | `partial` | `skip`
4. If not skipped: LLM-realize conversation + detection criteria
5. Write `runs/{id}.yaml`


## 1. Configuration

Pick an OpenAI-compatible provider. Ollama, OpenAI, Hugging Face Inference, and Claude (via OpenRouter or another OpenAI-compatible gateway) all work through the same client.

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Markdown

# Repo root = parent of examples/
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCENARIOS_DIR = REPO_ROOT / "examples" / "scenarios"
ARTIFACT_DIR = REPO_ROOT / "runs"

PROVIDER_PRESETS = {
    "ollama (local)": {
        "base_url": os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
        "model": os.environ.get("REDTEAM_MODEL", "qwen2.5:14b"),
        "api_key_env": None,
        "default_key": "ollama",
    },
    "openai": {
        "base_url": "https://api.openai.com/v1",
        "model": os.environ.get("REDTEAM_MODEL", "gpt-4o-mini"),
        "api_key_env": "OPENAI_API_KEY",
        "default_key": "",
    },
    "huggingface": {
        "base_url": os.environ.get(
            "HF_BASE_URL",
            "https://router.huggingface.co/v1",
        ),
        "model": os.environ.get("REDTEAM_MODEL", "meta-llama/Meta-Llama-3-8B-Instruct"),
        "api_key_env": "HF_TOKEN",
        "default_key": "",
    },
    "claude (via OpenRouter)": {
        "base_url": "https://openrouter.ai/api/v1",
        "model": os.environ.get("REDTEAM_MODEL", "anthropic/claude-3.5-sonnet"),
        "api_key_env": "OPENROUTER_API_KEY",
        "default_key": "",
    },
}

provider_dd = widgets.Dropdown(
    options=list(PROVIDER_PRESETS.keys()),
    description="Provider:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
)
base_url_tb = widgets.Text(
    description="Base URL:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="520px"),
)
model_tb = widgets.Text(
    description="Model:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="520px"),
)
api_key_tb = widgets.Password(
    description="API key:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="520px"),
)


def _apply_provider(*_):
    preset = PROVIDER_PRESETS[provider_dd.value]
    base_url_tb.value = preset["base_url"]
    model_tb.value = preset["model"]
    env_name = preset["api_key_env"]
    if env_name and os.environ.get(env_name):
        api_key_tb.value = os.environ[env_name]
    elif os.environ.get("OPENAI_API_KEY"):
        api_key_tb.value = os.environ["OPENAI_API_KEY"]
    else:
        api_key_tb.value = preset["default_key"]


provider_dd.observe(_apply_provider, names="value")
_apply_provider()
display(provider_dd, base_url_tb, model_tb, api_key_tb)
print(f"Repo root: {REPO_ROOT}")
print(f"Scenarios: {SCENARIOS_DIR}")

## 2. Select Scenario

Choose a bundled scenario and an exploit style from `exploit_styles.json`.

In [ ]:
from build_spec import list_style_names

scenario_files = sorted(p.name for p in SCENARIOS_DIR.glob("*.yaml"))
print(f"Found {len(scenario_files)} scenarios\n")

scenario_dd = widgets.Dropdown(
    options=scenario_files,
    description="Scenario:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
)
exploit_dd = widgets.Dropdown(
    options=["(random wording pack)"] + list_style_names(),
    value="embedded_instruction",
    description="Exploit:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
)

display(scenario_dd, exploit_dd)


def get_scenario_path() -> Path:
    return SCENARIOS_DIR / scenario_dd.value


## 3. Setup

Apply LLM settings, then import the Garak artifact path helpers.

In [ ]:
from spec_io import SPEC_FILE, default_run_dir
from build_spec import build_environment_spec, load_scenario, summarize_scenario
from asago_artifact_generator.garak.gate import gate_garak
from asago_artifact_generator.garak.realize import configure_llm, realize
from asago_artifact_generator.garak.spec_io import realized_to_artifact, save_artifact
import yaml

configure_llm(
    base_url=base_url_tb.value.strip(),
    api_key=api_key_tb.value.strip() or "ollama",
    model=model_tb.value.strip(),
)
print(f"LLM base URL: {base_url_tb.value.strip()}")
print(f"Model:        {model_tb.value.strip()}")
print(f"Artifact dir: {ARTIFACT_DIR}")

## 4. Label + Generate Garak Artifact

Runs:
1. **Load / build** schema-v2 `EnvironmentSpec` (writes `runs/{id}/spec.json`)
2. **Label** with `gate_garak` → `full` | `partial` | `skip`
3. **Realize** chat history + detector rubrics (LLM generated) 
4. **Write** `runs/{id}.yaml`

In [ ]:
scenario_path = get_scenario_path()
exploit_name = None if exploit_dd.value == "(random wording pack)" else exploit_dd.value

loaded = load_scenario(scenario_path)
print("=== Scenario ===")
print(summarize_scenario(loaded))
print()

spec, spec_source, _result = build_environment_spec(
    scenario_path,
    exploit_name=exploit_name,
    persist=True,
    force=True,
)
from spec_io import default_run_dir as _run_dir
spec_path = _run_dir(spec.scenario_id) / SPEC_FILE
print(f"EnvironmentSpec: {spec_path} (source={spec_source})")
print(f"  surface={spec.injection_surface}  oracle={spec.oracle_target}")
print(f"  exploit={spec.attack.exploit.name}")
print()

label, gated_spec, reason = gate_garak(spec)
print("=== Garak coverage label ===")
print(f"  label:  {label}")
print(f"  reason: {reason}")

artifact_path = None
realized = None

if label == "skip":
    print("\nSkipped — no Garak YAML written.")
else:
    print("\nRealizing conversation + detector rubrics via LLM...")
    realized = realize(gated_spec, use_llm=True)
    artifact_path = save_artifact(realized, ARTIFACT_DIR)
    print(f"Wrote Garak artifact: {artifact_path}")

## 5. Inspect Artifact

Chat history (`prompts`) is the probe input. `detection.oracle_predicates` are narrative rubrics suitable for LLM-as-a-judge.

In [ ]:
if label == "skip":
    display(Markdown(f"**Skipped (`{label}`):** {reason}"))
elif realized is None:
    display(Markdown("Run the previous cell first."))
else:
    cfg = realized_to_artifact(realized)
    display(Markdown(f"### Label: `{label}` — {reason}"))
    display(Markdown("### Chat history (`prompts`)"))
    print(cfg["prompts"][0])
    print()
    display(Markdown("### Detector / LLM-judge rubrics (`oracle_predicates`)"))
    for i, pred in enumerate(cfg["detection"]["oracle_predicates"], 1):
        print(f"{i}. {pred}")
    print()
    display(Markdown("### Programmatic predicates"))
    for p in cfg["detection"]["predicates"]:
        print(f"- {p}")
    print()
    display(Markdown("### Full YAML preview"))
    print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

## 6. Output Paths

Use the written artifact with Garak (after symlinking the Scenario plugins — see `README.md`).

In [ ]:
print(f"Scenario:     {scenario_path}")
print(f"Label:        {label} ({reason})")
print(f"Spec:         {spec_path}")
print(f"Garak YAML:   {artifact_path or '(none — skipped)'}")
if artifact_path:
    print()
    print("# Smoke-run example:")
    print(f'export SCENARIO_CONFIG="{artifact_path}"')
    print(
        "python -m garak --target_type test.Blank "
        "--probes scenario.Scenario --generations 1"
    )